# Cart-pendulum: automated sweep vs simulation

**Order of use:** upload `cart_sweep.ino` once → close the Arduino Serial Monitor → run the cells top to bottom.
Cart at the start end of the track, pendulum hanging still.

Cells: config → helper functions → quick one-trial test → full sweep → simulation → comparison plot.
You can re-run the simulation and plot cells without the hardware by loading a saved CSV
(`data = pd.read_csv("sweep_....csv")`).

In [ ]:
# Only needed once, if "import serial" fails. Installs pyserial into THIS notebook's Python; then restart the kernel.
# import subprocess, sys
# subprocess.check_call([sys.executable, "-m", "pip", "install", "pyserial"])

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, trapezoid
import serial                      # pip install pyserial

# ======================= CONFIG =======================
PORT = "COM8"
BAUD = 115200

# ---- Cart (shared by the experiment AND the simulation) ----
V_MAX_MM_S  = 163.43               # max cart speed [mm/s]            (from your sim notebook)
A_MAX_MM_S2 = 2621.03              # max cart acceleration [mm/s^2]   (from your sim notebook)

MOTOR_STEPS_PER_REV = 1600.0       # must match the .ino
STEPS_PER_MM = 43.24               # 265 rpm <-> 163.43 mm/s and 7277.292 steps = 168.3 mm both give 43.24.
                                   # VERIFY with a ruler: RUN,2000 should move the cart ~46.3 mm.
MAX_TRAVEL_STEPS = 8000            # must match MAX_TRAVEL_STEPS in the .ino (8000 steps = ~185 mm)

# Motor settings are DERIVED from the mm values, so experiment and simulation can't disagree
VMAX_RPM    = V_MAX_MM_S  * STEPS_PER_MM * 60.0 / MOTOR_STEPS_PER_REV     # ~265 rpm
ACCEL_RPM_S = A_MAX_MM_S2 * STEPS_PER_MM * 60.0 / MOTOR_STEPS_PER_REV     # ~4250 rpm/s
RETURN_RPM, RETURN_ACCEL_RPM_S = VMAX_RPM, ACCEL_RPM_S                    # return move (proven settings)
WINDOW_MS = 2000                   # how long after the stop the peak is measured (experiment AND simulation)
COOLDOWN_S = 15                    # pause between trials so the driver/motor/power bank can recover (0 = none)

# ---- Sweep ----
D_START_MM = 5
D_STOP_MM  = 180                   # to reach 250 mm: raise MAX_TRAVEL_STEPS in the .ino to >= 250*STEPS_PER_MM
                                   # (~10810) -- ONLY if the track really has that much travel
N_POINTS   = 13
REPEATS    = 2                     # 2-3 gives error bars
SHUFFLE    = False

# ---- Pendulum model (only these two numbers describe the pendulum) ----
OMEGA0 = 9.44                      # natural frequency [rad/s]. 9.44 reproduces the null positions of your simulated table;
                                   # replace with your measured value (frequency test) if it differs
ZETA   = 0.01                      # damping ratio              <-- tune until the sim's decay matches the rig
G      = 9.81

print(f"Using VMAX_RPM = {VMAX_RPM:.1f}, ACCEL_RPM_S = {ACCEL_RPM_S:.0f}")
print(f"Triangle/trapezoid boundary D = v_max^2/a_max = {V_MAX_MM_S**2/A_MAX_MM_S2:.1f} mm")

In [ ]:
def _read_until(ser, prefixes, timeout_s):
    """Read lines until one starts with one of `prefixes`. Raises on ERR or timeout."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        line = ser.readline().decode(errors="ignore").strip()
        if not line:
            continue
        if line.startswith("ERR"):
            raise RuntimeError(f"Arduino reported: {line}")
        if line.startswith(prefixes):
            return line
    raise TimeoutError(f"No {prefixes} reply within {timeout_s} s")


def _send(ser, text, expect, timeout_s=5):
    ser.write((text + "\n").encode())
    return _read_until(ser, expect, timeout_s)


def run_sweep(distances_mm, steps_per_mm=None, repeats=1, shuffle=False,
              port=None, baud=None, vmax_rpm=None, accel_rpm_s=None,
              window_ms=None, return_rpm=None, return_accel_rpm_s=None,
              trial_timeout_s=180, csv_path=None, cooldown_s=None):
    """Run every distance automatically and return a DataFrame of results (defaults come from CONFIG)."""
    steps_per_mm       = steps_per_mm       or STEPS_PER_MM
    port               = port               or PORT
    baud               = baud               or BAUD
    vmax_rpm           = vmax_rpm           or VMAX_RPM
    accel_rpm_s        = accel_rpm_s        or ACCEL_RPM_S
    window_ms          = window_ms          or WINDOW_MS
    return_rpm         = return_rpm         or RETURN_RPM
    return_accel_rpm_s = return_accel_rpm_s or RETURN_ACCEL_RPM_S
    cooldown_s         = COOLDOWN_S if cooldown_s is None else cooldown_s

    longest_steps = max(distances_mm) * steps_per_mm
    if longest_steps > MAX_TRAVEL_STEPS:
        raise ValueError(f"{max(distances_mm)} mm = {longest_steps:.0f} steps, more than MAX_TRAVEL_STEPS "
                         f"({MAX_TRAVEL_STEPS}). Lower D_STOP_MM, or raise MAX_TRAVEL_STEPS here AND in the .ino "
                         f"(only if the track allows it).")

    plan = [(float(d), r) for r in range(repeats) for d in distances_mm]
    if shuffle:
        np.random.shuffle(plan)

    if csv_path is None:
        csv_path = time.strftime("sweep_%Y%m%d_%H%M%S.csv")

    rows = []
    ser = serial.Serial(port, baud, timeout=1)
    try:
        time.sleep(2.0)                       # Arduino resets when the port opens
        ser.reset_input_buffer()
        _send(ser, "PING", "READY")

        for key, val in [("VMAX_RPM", vmax_rpm), ("ACCEL_RPM_S", accel_rpm_s),
                         ("WINDOW_MS", window_ms), ("RETURN_RPM", return_rpm),
                         ("RETURN_ACCEL_RPM_S", return_accel_rpm_s)]:
            _send(ser, f"SET,{key},{val}", "OK")

        for i, (d_mm, rep) in enumerate(plan, 1):
            steps = int(round(d_mm * steps_per_mm))
            t0 = time.time()
            reply = _send(ser, f"RUN,{steps}", "RESULT", timeout_s=trial_timeout_s)
            _, st, pos, neg, amp, settled = reply.split(",")

            rows.append(dict(D_mm=int(st) / steps_per_mm, steps=int(st), repeat=rep,
                             peak_pos_deg=float(pos), peak_neg_deg=float(neg),
                             amp_deg=float(amp), settled=bool(int(settled))))
            pd.DataFrame(rows).to_csv(csv_path, index=False)   # save after every trial

            flag = "" if rows[-1]["settled"] else "  (WARNING: pendulum did not fully settle)"
            print(f"[{i}/{len(plan)}] D = {d_mm:6.1f} mm -> amp = {amp} deg  ({time.time()-t0:.0f} s){flag}")
            if i < len(plan):
                time.sleep(cooldown_s)
    finally:
        ser.close()

    print(f"Saved: {csv_path}")
    return pd.DataFrame(rows).sort_values(["D_mm", "repeat"]).reset_index(drop=True)

## Quick test: ONE short trial first

In [ ]:
test = run_sweep([20])
test

## Run the sweep (the one button)

In [ ]:
distances_mm = np.linspace(D_START_MM, D_STOP_MM, N_POINTS)

data = run_sweep(distances_mm, repeats=REPEATS, shuffle=SHUFFLE)
data

## Built-in model (cross-check; parametrised only by ω₀ and ζ)

For a pendulum hanging from the cart, $\omega_0^2 = mgl/I$, so $ml/I = \omega_0^2/g$. The equation of motion
$\ddot\theta + (ml/I)\cos\theta\,\ddot x + \omega_0^2\sin\theta + 2\zeta\omega_0\dot\theta = 0$ therefore needs only the measured
$\omega_0$ and a damping ratio, with no mass or length guesses. The cart follows the same accelerate / cruise / decelerate
profile as the Arduino's AccelStepper move, and the peak is taken over the same `WINDOW_MS` after the stop as in the experiment.

In [ ]:
A_MAX = A_MAX_MM_S2 / 1000.0       # m/s^2
V_MAX = V_MAX_MM_S  / 1000.0       # m/s
K = OMEGA0**2 / G                  # = m*l/I


def motion_profile(D):
    """Accelerate / cruise / decelerate timing for target distance D [m]."""
    d_ramp = V_MAX**2 / A_MAX
    if D >= d_ramp:                          # trapezoid
        v_peak, t_acc, t_cruise = V_MAX, V_MAX / A_MAX, (D - d_ramp) / V_MAX
    else:                                    # triangle
        v_peak = np.sqrt(A_MAX * D); t_acc = v_peak / A_MAX; t_cruise = 0.0
    return dict(D=D, t_acc=t_acc, t_cruise=t_cruise, T=2*t_acc + t_cruise)


def cart_accel(t, p):
    if t < p["t_acc"]:                       return  A_MAX
    if t < p["t_acc"] + p["t_cruise"]:       return  0.0
    if t < p["T"]:                           return -A_MAX
    return 0.0


def rhs(t, s, p, linear):
    theta, omega = s
    xdd = cart_accel(t, p)
    damp = 2 * ZETA * OMEGA0 * omega
    if linear:
        alpha = -K*xdd - OMEGA0**2*theta - damp
    else:
        alpha = -K*np.cos(theta)*xdd - OMEGA0**2*np.sin(theta) - damp
    return [omega, alpha]


def sim_peak_deg(D_mm, linear):
    """Peak |theta| [deg] within WINDOW_MS after the cart stops."""
    p = motion_profile(D_mm / 1000.0)
    t_end = p["T"] + WINDOW_MS / 1000.0
    t = np.arange(0, t_end, 0.002)
    sol = solve_ivp(rhs, (0, t_end), [0.0, 0.0], args=(p, linear), t_eval=t,
                    max_step=0.002, rtol=1e-8, atol=1e-10)
    return np.degrees(np.max(np.abs(sol.y[0][sol.t >= p["T"]])))


def analytic_peak_deg(D_mm):
    """Linear, undamped prediction of the leftover amplitude [deg]."""
    p = motion_profile(D_mm / 1000.0)
    tt = np.linspace(0, p["T"], 20001)
    ax = np.array([cart_accel(ti, p) for ti in tt])
    return np.degrees(OMEGA0 / G * abs(trapezoid(ax * np.exp(1j * OMEGA0 * tt), tt)))


D_sim = np.linspace(5, 250, 40)
model_sim = pd.DataFrame({
    "D_mm":      D_sim,
    "nonlinear": [sim_peak_deg(d, linear=False) for d in D_sim],
    "linear":    [sim_peak_deg(d, linear=True)  for d in D_sim],
    "analytic":  [analytic_peak_deg(d) for d in D_sim],
})
model_sim.round(2)

## Your simulated data (D [mm] vs peak angle [deg])

Pasted from your simulation. To use a different file instead, set `SIM_CSV = "file.csv"` (two columns: distance in mm, angle in deg).

In [ ]:
SIM_CSV = None   # e.g. "my_sim.csv" with columns D_mm, angle_deg  (None = use the table below)

if SIM_CSV:
    table_sim = pd.read_csv(SIM_CSV, names=["D_mm", "simulation"], header=0)
else:
    _D = [5, 11.3, 17.6, 23.8, 30.1, 36.4, 42.7, 49, 55.3, 61.5, 67.8, 74.1, 80.4, 86.7, 92.9, 99.2, 105.5, 111.8,
          118.1, 124.4, 130.6, 136.9, 143.2, 149.5, 155.8, 162.1, 168.3, 174.6, 180.9, 187.2, 193.5, 199.7, 206,
          212.3, 218.6, 224.9, 231.2, 237.4, 243.7, 250]
    _A = [3.91, 8.67, 13.16, 17.22, 20.71, 23.52, 25.56, 26.76, 27.09, 26.52, 25.08, 22.81, 19.81, 16.15, 11.96,
          7.37, 2.55, 2.36, 7.19, 11.78, 16, 19.68, 22.71, 25, 26.48, 27.08, 26.78, 25.61, 23.61, 20.82, 17.35,
          13.32, 8.85, 4.08, 0.82, 5.69, 10.37, 14.72, 18.57, 21.83]
    table_sim = pd.DataFrame({"D_mm": _D, "simulation": _A})

# How does your table compare with the built-in model (same omega0, coupling m*l/I = omega0^2/g)?
ratio = table_sim.simulation.values / np.interp(table_sim.D_mm, model_sim.D_mm, model_sim.linear)
mask = table_sim.simulation.values > 3          # skip points near a null, where the ratio is noisy
print(f"Your table / built-in linear model (omega0 = {OMEGA0} rad/s): median ratio = {np.median(ratio[mask]):.2f}")

## Measured vs simulated

In [ ]:
SHOW_MODEL = True    # also draw the built-in model (nonlinear) and the analytic line

g = data.groupby("D_mm")["amp_deg"]
mean, std = g.mean(), g.std().fillna(0)

fig, ax = plt.subplots(figsize=(8.5, 5.8))
ax.plot(table_sim.D_mm, table_sim.simulation, "o-", ms=4, color="C0", label="simulation (your table)")
if SHOW_MODEL:
    ax.plot(model_sim.D_mm, model_sim.nonlinear, "s--", ms=3, color="C1", label="built-in model, nonlinear")
    ax.plot(model_sim.D_mm, model_sim.analytic,  "k:",  label="built-in model, analytic (linear, undamped)")
ax.errorbar(mean.index, mean.values, yerr=std.values if data["repeat"].nunique() > 1 else None,
            fmt="D-", color="C3", capsize=3, label="experiment")

d_crit = V_MAX_MM_S**2 / A_MAX_MM_S2
ax.axvline(d_crit, color="gray", ls="--", lw=0.8)
ax.text(d_crit + 2, ax.get_ylim()[1] * 0.5, "D = v_max^2/a_max", rotation=90, color="gray", fontsize=8, va="center")

ax.set_xlabel("target distance D [mm]")
ax.set_ylabel("peak |theta| after stop [deg]")
ax.set_title("Oscillation amplitude after stopping vs distance D")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
plt.savefig("measured_vs_sim.png", dpi=150, bbox_inches="tight")
plt.show()

# Numbers: RMS difference over the distances the experiment covered
for name, x, y in [("your table", table_sim.D_mm, table_sim.simulation),
                   ("built-in model (nonlinear)", model_sim.D_mm, model_sim.nonlinear)]:
    rms = np.sqrt(np.mean((mean.values - np.interp(mean.index, x, y)) ** 2))
    print(f"RMS difference, experiment vs {name}: {rms:.2f} deg")